# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading and exploring the FAIR^2 clinical dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the URL for the Croissant schema
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Review available record sets, fields, and their IDs.

### Note
* All record sets, fields, and columns are referenced explicitly by their `@id` fields for clarity and reproducibility, per FAIR^2 and Croissant best practices.

In [ ]:
# Overview: list available record sets and their fields (by @id)
record_sets = list(dataset.metadata.recordSet)
print("Available Record Sets (@id):")
for rs in record_sets:
    print(f"- {rs['@id']} ({rs.get('name', '[no name]')})")

# For each record set, list fields and columns by @id
for rs in record_sets:
    print(f"\nRecordSet: {rs['@id']} ({rs.get('name', '[no name]')})")
    fields = rs.get('field', [])
    if not fields:
        print("  No fields found.")
    else:
        print("  Fields (@id):")
        for fld in fields:
            print(f"    - {fld['@id']} (name: {fld.get('name', '[no name]')})")
            columns = fld.get('column', [])
            if columns:
                print("      Columns (@id):")
                for col in columns:
                    print(f"        - {col['@id']} (name: {col.get('name', '[no name]')})")

## 3. Data Extraction

Load records from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Collect all record set @ids
record_set_ids = [rs['@id'] for rs in record_sets]

# Load dataframes for each record set
dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    # convert to DataFrame
    df = pd.DataFrame(records)
    dataframes[rs_id] = df

# Print the columns for the first record set
if record_set_ids:
    primary_record_set_id = record_set_ids[0]
    print(f"Columns in record set {primary_record_set_id}:")
    print(dataframes[primary_record_set_id].columns.tolist())
    dataframes[primary_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

Apply data processing steps, such as filtering records by criteria, normalizing numeric fields, categorizing, transforming, and grouping data.

In [ ]:
# Example: choose a numeric column @id from primary record set

# Suppose the column @id for 'Age at Second CRC diagnosis' is available as follows (adjust if actual @id differs)
numeric_field_id = None
group_field_id = None
for rs in record_sets:
    if rs['@id'] == primary_record_set_id:
        for fld in rs.get('field', []):
            if 'age' in (fld.get('name', '').lower()):
                numeric_field_id = fld['@id']
            if 'sex' in (fld.get('name', '').lower()):
                group_field_id = fld['@id']

df = dataframes[primary_record_set_id]
# If actual @id not found, fallback to a column present in DataFrame
if numeric_field_id and numeric_field_id in df.columns:
    numeric_field_col = numeric_field_id
else:
    # fallback: pick a numeric column
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    numeric_field_col = numeric_cols[0] if numeric_cols else df.columns[0]

if group_field_id and group_field_id in df.columns:
    group_col = group_field_id
else:
    possible_groups = [col for col in df.columns if any(x in col.lower() for x in ['sex','site','location','msi'])]
    group_col = possible_groups[0] if possible_groups else None

# Example EDA: filter records where numeric column > threshold
threshold = 50
filtered_df = df[df[numeric_field_col] > threshold]
print(f"Filtered records with {numeric_field_col} > {threshold}:")
print(filtered_df.head())

# Normalize this column
filtered_df[f"{numeric_field_col}_normalized"] = (filtered_df[numeric_field_col] - filtered_df[numeric_field_col].mean()) / filtered_df[numeric_field_col].std()
print(f"Normalized {numeric_field_col} for filtered records:")
print(filtered_df[[numeric_field_col, f"{numeric_field_col}_normalized"]].head())

# Group by the group field (e.g., sex), if present
if group_col and group_col in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_col)[numeric_field_col].mean().reset_index()
    print(f"Grouped data by {group_col} (mean {numeric_field_col}):")
    print(grouped_df.head())

## 5. Visualization

Visualize the distribution of a key numeric column and relationships with group column.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram for numeric column
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field_col].dropna(), bins=10, kde=True)
plt.title(f'Distribution of {numeric_field_col}')
plt.xlabel(numeric_field_col)
plt.ylabel('Count')
plt.show()

# If group column available, plot boxplot
if group_col and group_col in df.columns:
    plt.figure(figsize=(8,4))
    sns.boxplot(x=group_col, y=numeric_field_col, data=df)
    plt.title(f'{numeric_field_col} by {group_col}')
    plt.show()

## 6. Conclusion

In this notebook, we loaded the FAIR^2 clinical oncology dataset via Croissant, explored its record sets and fields using their `@id` identifiers, extracted records into DataFrames, applied common EDA and processing steps (filtering, normalization, grouping), and visualized relationships in the data.

**Key insights:**
- Structure and quality of clinical records are accessible via Croissant and can be programmatically parsed.
- Numeric variables (e.g., age) can be filtered and normalized to support downstream clinical analytics.
- Grouping and visualization enables comparative investigation, e.g., of anatomical or molecular biomarkers by sex or MSI status.

This template is extensible for further FAIR^2 dataset analyses and reproducibility. For details, consult [mlcroissant documentation](https://github.com/mlcommons/croissant).